# 🫀 Intelligent ECG Analysis Tool
### Signal-to-Report and Signal-to-Diagnosis with Deep Learning

**Pipeline overview:**
```
ECG Signal → [Frozen HuBERT-ECG Encoder] → features
                                           ├─→ [Classifier Head]   → Label + Confidence
                                           └─→ [Adapter + BART]   → Clinical Report
```

**Steps covered in this notebook:**
| Step | Description |
|------|-------------|
| 1 | Environment & data setup (PTB-XL) |
| 2 | Load frozen HuBERT-ECG encoder, extract features |
| 3 | Train classifier head (signal → label) |
| 4 | Train report generator (signal → text) |
| 5 | Launch Gradio web app |
| 6 | Systematic evaluation & error analysis |

> ⚠️ **Before running:** add the PTB-XL dataset to this notebook via  
> *Add Data → search "PTB-XL: A Large Publicly Available ECG Dataset"*


## 📦 Step 0 — Install Dependencies
*(runs once per Kaggle session, ~5 min)*

In [ ]:
# Install all required packages
# The HuBERT-ECG line installs the custom model class from GitHub
import subprocess, sys

packages = [
    "torch>=2.1.0",
    "torchaudio>=2.1.0",
    "transformers>=4.40.0",
    "wfdb>=4.1.2",
    "numpy>=1.24.0",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "scikit-learn>=1.3.0",
    "gradio>=4.30.0",
    "tqdm>=4.66.0",
    "evaluate>=0.4.1",
    "rouge-score>=0.1.2",
    "nltk>=3.8.1",
    "peft>=0.11.0",
    "huggingface_hub>=0.23.0",
    "scipy>=1.11.0",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

# HuBERT-ECG custom package MUST be installed BEFORE importing transformers
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/Edoar-do/HuBERT-ECG.git"
])

print("✔  All packages installed.")


## ⚙️ Configuration
*(edit `PTBXL_DIR` if your dataset path differs)*

In [ ]:
import os
import torch

# ── Paths ──────────────────────────────────────────────────────────
# Kaggle mounts PTB-XL datasets under /kaggle/input/
# The folder name varies; we auto-detect it below.
def _find_ptbxl():
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    for entry in os.listdir(base):
        lower = entry.lower()
        if "ptb" in lower and ("xl" in lower or "ecg" in lower):
            candidate = os.path.join(base, entry)
            if os.path.exists(os.path.join(candidate, "ptbxl_database.csv")):
                return candidate
    return None

PTBXL_DIR = _find_ptbxl() or os.path.join(os.getcwd(), "data", "ptbxl")
print(f"PTB-XL directory: {PTBXL_DIR}")
assert os.path.exists(os.path.join(PTBXL_DIR, "ptbxl_database.csv")),             "❌ ptbxl_database.csv not found! Add the PTB-XL dataset via 'Add Data'."

CHECKPOINT_DIR     = "/kaggle/working/checkpoints"
OUTPUT_DIR         = "/kaggle/working/outputs"
FEATURES_CACHE_DIR = "/kaggle/working/features_cache"

for d in [CHECKPOINT_DIR, OUTPUT_DIR, FEATURES_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {DEVICE}")

# ── Signal params ──────────────────────────────────────────────────
SAMPLING_RATE       = 500   # Hz  (PTB-XL records500/)
SIGNAL_LENGTH_SEC   = 10
N_LEADS             = 12
PREPROCESSING_MODE  = "zscore"   # "zscore" | "minmax" | "bandpass_minmax"

# ── Encoder ────────────────────────────────────────────────────────
HUBERT_ECG_MODEL_ID           = "Edoardo-BS/hubert-ecg-base"
ENCODER_FEATURE_DIM           = 768
USE_FALLBACK_ENCODER_IF_UNAVAILABLE = True
FALLBACK_GUARD                = False  # set True to block training on random features

# ── Classifier ─────────────────────────────────────────────────────
SUPERCLASSES          = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES           = len(SUPERCLASSES)
CLASSIFIER_HIDDEN_DIM = 256
CLASSIFIER_LR         = 1e-3
CLASSIFIER_EPOCHS     = 30
CLASSIFIER_BATCH_SIZE = 64

# ── Report generator ───────────────────────────────────────────────
BART_MODEL_ID         = "facebook/bart-base"
ADAPTER_HIDDEN_DIM    = 512
MAX_REPORT_TOKENS     = 128
REPORT_GEN_LR         = 5e-5
REPORT_GEN_EPOCHS     = 10
REPORT_GEN_BATCH_SIZE = 8
REPORT_GEN_NUM_BEAMS  = 4
USE_LORA              = True

# ── Misc ───────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── `config` namespace ────────────────────────────────────────────
# src/*.py (the cells below) do `from src import config` and read
# `config.PTBXL_DIR`, `config.SAMPLING_RATE`, etc. Rather than
# duplicating every value under two names, wrap the settings above
# in a real `config` object so `config.X` keeps working unchanged
# (this mirrors src/config.py being an importable module there).
# NOTE: built from a snapshot of globals() taken *here*, after every
# setting above is assigned — it does not pick up unrelated
# uppercase names defined in later cells.
import types as _types
config = _types.SimpleNamespace(**{
    k: v for k, v in dict(globals()).items()
    if k.isupper() and not k.startswith("_")
})

print("✔  Configuration loaded.")


## 📊 Step 1 — Load PTB-XL Dataset

In [ ]:
import os
import ast
import argparse
import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt

# from src import config  # (removed by generate_notebook.py — already in notebook scope)


def load_ptbxl_metadata():
    """Load the main PTB-XL metadata CSV and parse SCP-code annotations."""
    csv_path = os.path.join(config.PTBXL_DIR, "ptbxl_database.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"Could not find {csv_path}.\n"
            "Download PTB-XL from https://physionet.org/content/ptb-xl/ and "
            f"unzip it into {config.PTBXL_DIR} (see README)."
        )
    df = pd.read_csv(csv_path, index_col="ecg_id")
    df.scp_codes = df.scp_codes.apply(ast.literal_eval)
    return df


def load_scp_statements():
    path = os.path.join(config.PTBXL_DIR, "scp_statements.csv")
    scp_df = pd.read_csv(path, index_col=0)
    scp_df = scp_df[scp_df.diagnostic == 1]
    return scp_df


def aggregate_diagnostic_superclass(scp_codes, scp_df):
    """Map raw SCP codes -> one or more of the 5 superclasses (NORM, MI, STTC, CD, HYP)."""
    classes = set()
    for code in scp_codes.keys():
        if code in scp_df.index:
            classes.add(scp_df.loc[code].diagnostic_class)
    return list(classes)


def build_label_matrix(df, scp_df):
    """Attach a multi-hot label vector (over config.SUPERCLASSES) to every record."""
    df = df.copy()
    df["superclasses"] = df.scp_codes.apply(lambda codes: aggregate_diagnostic_superclass(codes, scp_df))

    label_matrix = np.zeros((len(df), config.NUM_CLASSES), dtype=np.float32)
    for i, classes in enumerate(df["superclasses"]):
        for c in classes:
            if c in config.SUPERCLASSES:
                label_matrix[i, config.SUPERCLASSES.index(c)] = 1.0
    df["label_vector"] = list(label_matrix)
    return df


def get_record_path(row, sampling_rate=config.SAMPLING_RATE):
    """Return the on-disk path (without extension) for a given metadata row."""
    if sampling_rate == 100:
        rel = row.filename_lr
    else:
        rel = row.filename_hr
    return os.path.join(config.PTBXL_DIR, rel)


def load_raw_signal(row, sampling_rate=config.SAMPLING_RATE):
    """Load one 12-lead ECG as a (n_samples, 12) numpy array."""
    path = get_record_path(row, sampling_rate)
    signal, meta = wfdb.rdsamp(path)
    return signal, meta


def get_official_splits(df):
    """
    PTB-XL ships an official 10-fold split in `strat_fold` (1-10).
    Standard convention: fold 9 = validation, fold 10 = test, 1-8 = train.
    """
    train_df = df[df.strat_fold <= 8]
    val_df = df[df.strat_fold == 9]
    test_df = df[df.strat_fold == 10]
    return train_df, val_df, test_df


def load_full_dataset():
    """Convenience: metadata + scp statements + labels + official splits, all in one call."""
    df = load_ptbxl_metadata()
    scp_df = load_scp_statements()
    df = build_label_matrix(df, scp_df)
    train_df, val_df, test_df = get_official_splits(df)
    return {
        "full": df,
        "scp_df": scp_df,
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }


def plot_12_lead(signal, title="ECG", save_path=None):
    """Plot all 12 leads stacked vertically, PTB-XL lead order."""
    lead_names = ["I", "II", "III", "aVR", "aVL", "aVF",
                  "V1", "V2", "V3", "V4", "V5", "V6"]
    fig, axes = plt.subplots(12, 1, figsize=(10, 14), sharex=True)
    for i, ax in enumerate(axes):
        ax.plot(signal[:, i], linewidth=0.8, color="black")
        ax.set_ylabel(lead_names[i], rotation=0, labelpad=20, fontsize=9)
        ax.set_yticks([])
    axes[-1].set_xlabel("Samples")
    fig.suptitle(title)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f"Saved plot to {save_path}")
    else:
        plt.show()
    plt.close(fig)


def _explore():
    """CHECKPOINT for Step 1: load one record, plot it, print label + report text."""
    data = load_full_dataset()
    df = data["full"]
    row = df.iloc[0]

    signal, meta = load_raw_signal(row)
    print(f"Loaded record {row.name}: shape={signal.shape}, fs={meta['fs']}")
    print(f"Diagnostic superclasses: {row['superclasses']}")
    print(f"Label vector ({config.SUPERCLASSES}): {row['label_vector']}")
    print(f"Free-text cardiologist report: {row.get('report', 'N/A')}")

    train_df, val_df, test_df = get_official_splits(df)
    print(f"Splits -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

    out_path = os.path.join(config.OUTPUT_DIR, "example_ecg_plot.png")
    plot_12_lead(signal, title=f"Record {row.name}", save_path=out_path)



### ✅ Step 1 Checkpoint — Load & Plot One Record

In [ ]:
# Load full dataset metadata
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle

data = load_full_dataset()
df   = data["full"]
row  = df.iloc[0]

signal, meta = load_raw_signal(row)
print(f"Record {row.name}: shape={signal.shape}, fs={meta['fs']} Hz")
print(f"Diagnostic superclasses : {row['superclasses']}")
print(f"Label vector {SUPERCLASSES}: {row['label_vector']}")
print(f"Free-text report        : {row.get('report', 'N/A')}")

train_df, val_df, test_df = get_official_splits(df)
print(f"\nSplits → train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Plot and save
out_path = os.path.join(OUTPUT_DIR, "example_ecg_plot.png")
plot_12_lead(signal, title=f"Record {row.name}", save_path=out_path)

# Display inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
img = mpimg.imread(out_path)
plt.figure(figsize=(10, 14))
plt.imshow(img)
plt.axis('off')
plt.title("Step 1 Checkpoint: 12-lead ECG")
plt.show()
print("\n✔  Step 1 CHECKPOINT PASSED")


## 🧠 Step 2 — Load Frozen HuBERT-ECG Encoder

In [ ]:
"""
Step 2: Load the (frozen) HuBERT-ECG encoder and extract features.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HOW HuBERT-ECG LOADING WORKS (read this before training)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The real HuBERT-ECG uses a custom model class registered with
Hugging Face via the `hubert_ecg` pip package from the authors'
GitHub repo. You need to do TWO things before it loads:

  1. pip install git+https://github.com/Edoar-do/HuBERT-ECG.git
     (this is already in requirements.txt)

  2. import hubert_ecg  ← registers the custom class with AutoModel
     (this file does it automatically)

Expected input format for HuBERT-ECG:
  - Shape: (batch_size, n_leads=12, n_samples)
  - Sampling rate: 500 Hz  (10 s signal → 5000 samples)
  - Normalization: per-lead z-score (mean=0, std=1) by default, or bandpass + min-max [-1, 1]
    (configurable in config.py via PREPROCESSING_MODE)
  - The model config says it was pre-trained at 500 Hz.

Note on repository naming:
  - GitHub Repository URL: https://github.com/Edoar-do/HuBERT-ECG.git (owner: Edoar-do)
  - Hugging Face Hub Organization: Edoardo-BS (e.g., "Edoardo-BS/hubert-ecg-base")

  PTB-XL has both 100Hz and 500Hz versions. config.py sets
  SAMPLING_RATE=500 when using the real encoder so the signal
  length matches what the model was trained on.

Fallback path:
  If the real checkpoint cannot be loaded (network error, wrong ID,
  first run before install), a small untrained CNN+Transformer encoder
  is used. It produces valid tensor shapes so the rest of the pipeline
  can be tested, but quality will be poor — it is NOT pre-trained.

Step 2 Checkpoint (from Project_Plan.md §6.2):
  "A raw ECG goes in, a feature tensor of known shape comes out,
  and the encoder's parameters are confirmed frozen (zero gradients)."

Usage:
    python -m src.encoder --test
"""
import os
import numpy as np
import torch
import torch.nn as nn

# from src import config  # (removed by generate_notebook.py — already in notebook scope)


# ──────────────────────────────────────────────────────────────────────
# Fallback encoder (untrained — only for pipeline demos)
# ──────────────────────────────────────────────────────────────────────

class FallbackECGEncoder(nn.Module):
    """
    Small CNN + Transformer used ONLY if HuBERT-ECG cannot be loaded.
    NOT pre-trained — produces valid tensor shapes but random features.
    Swap in the real checkpoint (see README) before reporting any results.
    """

    def __init__(self, in_channels=config.N_LEADS,
                 feature_dim=config.ENCODER_FEATURE_DIM):
        super().__init__()
        self.feature_dim = feature_dim
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 64,  kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128,          kernel_size=9,  stride=2, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, feature_dim, kernel_size=5,  stride=2, padding=2),
            nn.BatchNorm1d(feature_dim),
            nn.GELU(),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim, nhead=8,
            dim_feedforward=feature_dim * 4,
            batch_first=True, dropout=0.1,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

    def forward(self, x):
        # x: (B, n_leads, T)
        h = self.conv(x)          # (B, feature_dim, T')
        h = h.transpose(1, 2)    # (B, T', feature_dim)
        h = self.transformer(h)  # (B, T', feature_dim)
        return h


# ──────────────────────────────────────────────────────────────────────
# Pre-processing helpers (Step 2, To-do item 3)
# ──────────────────────────────────────────────────────────────────────

def _bandpass_filter(signal: np.ndarray, lowcut: float = 0.5, highcut: float = 50.0, fs: float = 500.0) -> np.ndarray:
    """Apply Butterworth bandpass filter (0.5 - 50 Hz) across each lead."""
    try:
        from scipy.signal import butter, filtfilt
        nyq = 0.5 * fs
        low = max(1e-4, lowcut / nyq)
        high = min(0.9999, highcut / nyq)
        b, a = butter(N=3, Wn=[low, high], btype="band")
        filtered = np.zeros_like(signal)
        for ch in range(signal.shape[1]):
            filtered[:, ch] = filtfilt(b, a, signal[:, ch])
        return filtered
    except Exception:
        # Fall back gracefully to unfiltered signal if scipy bandpass fails
        return signal


def _minmax_normalize(signal: np.ndarray, feature_range: tuple = (-1, 1)) -> np.ndarray:
    """Scale signal per-lead to feature_range (default [-1, 1])."""
    min_val = signal.min(axis=0, keepdims=True)
    max_val = signal.max(axis=0, keepdims=True)
    diff = np.where((max_val - min_val) < 1e-8, 1.0, max_val - min_val)
    norm = (signal - min_val) / diff  # [0, 1]
    low, high = feature_range
    return (norm * (high - low) + low).astype(np.float32)


def preprocess_signal(signal: np.ndarray, mode: str = None) -> np.ndarray:
    """
    Preprocess raw ECG waveform (T, n_leads) for encoder input.

    Modes supported:
      - "zscore": Per-lead zero-mean unit-variance (mean=0, std=1). Standard default.
      - "minmax": Per-lead min-max scaling to [-1, 1].
      - "bandpass_minmax": 0.5-50Hz Butterworth bandpass filter + [-1, 1] min-max scaling.
                          Matches community HuBERT-ECG usage notebooks.

    Args:
        signal: (T, n_leads) numpy array — raw output from wfdb.rdsamp() or load_raw_signal()
        mode: Preprocessing mode string ("zscore", "minmax", "bandpass_minmax").
              If None, defaults to config.PREPROCESSING_MODE.

    Returns:
        (T, n_leads) numpy array (float32) — normalized waveform
    """
    if mode is None:
        mode = getattr(config, "PREPROCESSING_MODE", "zscore")

    if mode == "bandpass_minmax":
        fs = float(getattr(config, "SAMPLING_RATE", 500))
        filtered = _bandpass_filter(signal, fs=fs)
        return _minmax_normalize(filtered, feature_range=(-1, 1))
    elif mode == "minmax":
        return _minmax_normalize(signal, feature_range=(-1, 1))
    else:  # "zscore" default
        mean = signal.mean(axis=0, keepdims=True)   # (1, n_leads)
        std  = signal.std(axis=0,  keepdims=True)   # (1, n_leads)
        std  = np.where(std < 1e-8, 1.0, std)       # avoid div-by-zero for flat leads
        return ((signal - mean) / std).astype(np.float32)


def signal_to_tensor(signal: np.ndarray, device: str) -> torch.Tensor:
    """
    Convert (T, n_leads) numpy array → (1, n_leads, T) tensor on device.
    This is the input shape expected by HuBERT-ECG and the fallback CNN.
    """
    x = torch.tensor(signal, dtype=torch.float32)  # (T, n_leads)
    x = x.T.unsqueeze(0)                            # (1, n_leads, T)
    return x.to(device)


def batch_signals_to_tensor(signals: np.ndarray, device: str) -> torch.Tensor:
    """
    Convert (B, T, n_leads) numpy array → (B, n_leads, T) tensor on device.
    """
    x = torch.tensor(signals, dtype=torch.float32)  # (B, T, n_leads)
    x = x.permute(0, 2, 1)                          # (B, n_leads, T)
    return x.to(device)


# ──────────────────────────────────────────────────────────────────────
# Main encoder wrapper
# ──────────────────────────────────────────────────────────────────────

class ECGEncoder:
    """
    Unified wrapper around HuBERT-ECG (primary) or FallbackECGEncoder.

    Both expose the same interface:
        encode(signal)        → (L, d) tensor    [single signal]
        encode_batch(signals) → (B, L, d) tensor [batch]

    where L = number of time frames output by the encoder,
          d = feature dimension (768 for hubert-ecg-base).
    """

    def __init__(self, device=config.DEVICE):
        self.device      = device
        self.is_fallback = False
        self.model       = None
        self.feature_dim = config.ENCODER_FEATURE_DIM
        self._load()

    def _load(self):
        try:
            # Step 1 of the loading dance: register HuBERTECG with AutoModel
            # This import does nothing else — it just registers the class.
            try:
                import hubert_ecg  # noqa: F401  ← registers custom class
                print("hubert_ecg package imported (custom class registered).")
            except ImportError:
                print(
                    "[WARNING] hubert_ecg package not found. "
                    "Run:  pip install git+https://github.com/Edoar-do/HuBERT-ECG.git\n"
                    "Falling back to untrained CNN encoder."
                )
                raise  # triggers fallback

            from transformers import AutoModel
            print(f"Loading HuBERT-ECG from '{config.HUBERT_ECG_MODEL_ID}' …")
            self.model = AutoModel.from_pretrained(
                config.HUBERT_ECG_MODEL_ID,
                trust_remote_code=True,
            )
            self.feature_dim = getattr(
                self.model.config, "hidden_size", config.ENCODER_FEATURE_DIM
            )
            print(f"✔  HuBERT-ECG loaded. Feature dim = {self.feature_dim}")

        except Exception as e:
            if not config.USE_FALLBACK_ENCODER_IF_UNAVAILABLE:
                raise

            if config.FALLBACK_GUARD:
                raise RuntimeError(
                    "\n"
                    "══════════════════════════════════════════════════════════\n"
                    "  FALLBACK ENCODER GUARD TRIGGERED\n"
                    "══════════════════════════════════════════════════════════\n"
                    f"  Could not load HuBERT-ECG: {e}\n\n"
                    "  FALLBACK_GUARD = True in config.py, so we refuse to\n"
                    "  continue with the untrained CNN encoder during training.\n"
                    "  Training on random features would produce meaningless results.\n\n"
                    "  Fix options:\n"
                    "  1. Install the hubert_ecg package:\n"
                    "       pip install git+https://github.com/Edoar-do/HuBERT-ECG.git\n"
                    "  2. Make sure you have internet access on Kaggle/Colab.\n"
                    "  3. To test the pipeline shape without the real encoder, set\n"
                    "       FALLBACK_GUARD = False  in src/config.py\n"
                    "══════════════════════════════════════════════════════════\n"
                ) from e

            # FALLBACK_GUARD = False → warn loudly but continue
            print("\n" + "=" * 60)
            print("  ⚠  WARNING: Using UNTRAINED fallback encoder!")
            print(f"  Reason: {e}")
            print("  Results will NOT be meaningful until the real")
            print("  HuBERT-ECG checkpoint is loaded.")
            print("=" * 60 + "\n")
            self.model       = FallbackECGEncoder()
            self.is_fallback = True

        self.model.to(self.device)
        self.model.eval()
        # Freeze all parameters — the encoder is used as a fixed feature extractor
        for p in self.model.parameters():
            p.requires_grad = False

    # ── public API ─────────────────────────────────────────────────────

    @torch.no_grad()
    def encode(self, signal: np.ndarray) -> torch.Tensor:
        """
        Step 2, To-do item 4: pass one ECG signal through the encoder.

        Args:
            signal: (T, n_leads) numpy array from wfdb.rdsamp() or data.load_raw_signal()
                    T = SIGNAL_LENGTH_SEC * SAMPLING_RATE  (e.g. 10 * 500 = 5000)
        Returns:
            (L, d) torch tensor — frozen feature representation
        """
        signal = preprocess_signal(signal)       # configurable preprocessing (see config.PREPROCESSING_MODE)
        x      = signal_to_tensor(signal, self.device)  # (1, n_leads, T)

        if self.is_fallback:
            out = self.model(x)                       # (1, L, d)
        else:
            out = self.model(x).last_hidden_state     # HF models expose this

        return out.squeeze(0).cpu()  # (L, d)

    @torch.no_grad()
    def encode_batch(self, signals: np.ndarray) -> torch.Tensor:
        """
        Step 2, To-do item 5: encode a batch of signals.

        Args:
            signals: (B, T, n_leads) numpy array
        Returns:
            (B, L, d) torch tensor
        """
        signals = np.stack([preprocess_signal(s) for s in signals])  # normalize each
        x       = batch_signals_to_tensor(signals, self.device)       # (B, n_leads, T)

        if self.is_fallback:
            out = self.model(x)
        else:
            out = self.model(x).last_hidden_state
        return out.cpu()

    def confirm_frozen(self) -> bool:
        """Returns True if ALL encoder parameters have requires_grad=False."""
        return all(not p.requires_grad for p in self.model.parameters())


# ──────────────────────────────────────────────────────────────────────
# Step 2 checkpoint test  (run with:  python -m src.encoder --test)
# ──────────────────────────────────────────────────────────────────────

def _test():
    """
    Full Step 2 checkpoint from Project_Plan.md §6.2:

    1. Load the encoder + confirm frozen (To-do 1 & 2)
    2. Preprocess a real PTB-XL ECG signal (To-do 3)
    3. Pass it through → inspect output shape (To-do 4)
    4. Encode a small BATCH and save features to disk (To-do 5)
    """
    import os
    import numpy as np
    from src.data import load_full_dataset, load_raw_signal

    print("\n" + "=" * 60)
    print("  Step 2 Checkpoint — Encoder Feature Extraction")
    print("=" * 60 + "\n")

    # ── 1 & 2: Load encoder, confirm frozen ───────────────────────────
    enc = ECGEncoder()
    frozen = enc.confirm_frozen()
    print(f"  Encoder frozen (zero gradients): {frozen}")
    print(f"  Using fallback encoder         : {enc.is_fallback}")
    print(f"  Feature dimension (d)          : {enc.feature_dim}")
    assert frozen, "FAIL: encoder parameters are not frozen!"

    # ── 3 & 4: Preprocess + single forward pass ────────────────────────
    print("\n  Loading one real ECG from PTB-XL …")
    data   = load_full_dataset()
    row    = data["test"].iloc[0]
    signal, meta = load_raw_signal(row)
    print(f"  Raw signal shape  : {signal.shape}   (T={signal.shape[0]}, leads={signal.shape[1]})")
    print(f"  Sampling rate     : {meta['fs']} Hz")

    # Apply preprocessing (per-lead z-score)
    signal_norm = preprocess_signal(signal)
    print(f"  After normalization: mean={signal_norm.mean():.4f}, std={signal_norm.std():.4f}")

    feats = enc.encode(signal)
    print(f"\n  ✔ Feature tensor shape (L × d): {tuple(feats.shape)}")
    print(f"     L = {feats.shape[0]} time frames")
    print(f"     d = {feats.shape[1]} feature dimensions")

    # ── 5: Batch encoding + save to disk ──────────────────────────────
    print("\n  Encoding a small batch of 4 ECGs …")
    batch_rows    = [data["test"].iloc[i] for i in range(4)]
    batch_signals = np.stack([load_raw_signal(r)[0] for r in batch_rows])  # (4, T, 12)
    batch_feats   = enc.encode_batch(batch_signals)
    print(f"  ✔ Batch feature shape (B × L × d): {tuple(batch_feats.shape)}")

    # Save the 4 feature arrays to disk for inspection
    save_dir = os.path.join(config.OUTPUT_DIR, "step2_feature_samples")
    os.makedirs(save_dir, exist_ok=True)
    for i, row in enumerate(batch_rows):
        np.save(os.path.join(save_dir, f"features_ecg_{row.name}.npy"),
                batch_feats[i].numpy())
    print(f"  ✔ Saved {len(batch_rows)} feature files to {save_dir}/")

    print("\n" + "=" * 60)
    print("  CHECKPOINT PASSED ✔")
    print("  A raw ECG went in, a feature tensor of known shape came out.")
    print(f"  Encoder frozen: {frozen}")
    print("=" * 60 + "\n")



### ✅ Step 2 Checkpoint — Feature Extraction

In [ ]:
import numpy as np

enc = ECGEncoder()

frozen = enc.confirm_frozen()
print(f"Encoder frozen (zero gradients): {frozen}")
print(f"Using fallback encoder         : {enc.is_fallback}")
print(f"Feature dimension (d)          : {enc.feature_dim}")
assert frozen, "FAIL: encoder not frozen!"

# Single signal
row    = data["test"].iloc[0]
signal, meta = load_raw_signal(row)
feats  = enc.encode(signal)
print(f"\nSingle signal → feature shape (L × d): {tuple(feats.shape)}")

# Batch
batch_rows    = [data["test"].iloc[i] for i in range(4)]
batch_signals = np.stack([load_raw_signal(r)[0] for r in batch_rows])
batch_feats   = enc.encode_batch(batch_signals)
print(f"Batch (4) feature shape (B × L × d)  : {tuple(batch_feats.shape)}")

print("\n✔  Step 2 CHECKPOINT PASSED")


## 🗄️ Dataset Utilities (Feature Caching)

In [ ]:
"""
PyTorch Dataset wrappers used by both the classifier (Step 3) and the
report generator (Step 4). Features are extracted once through the frozen
encoder and cached to disk (data/features_cache/) so repeated epochs don't
re-run the encoder forward pass every time.
"""
import os
import hashlib
import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

# from src import config  # (removed by generate_notebook.py — already in notebook scope)
# from src.data import load_raw_signal  # (removed by generate_notebook.py — already in notebook scope)


def _cache_path(ecg_id):
    return os.path.join(config.FEATURES_CACHE_DIR, f"{ecg_id}.npy")


def precompute_features(df, encoder, desc="Extracting features"):
    """Run every record in df through the frozen encoder once, caching to disk."""
    for ecg_id, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        cpath = _cache_path(ecg_id)
        if os.path.exists(cpath):
            continue
        signal, _ = load_raw_signal(row)
        feats = encoder.encode(signal)  # (L, d)
        np.save(cpath, feats.numpy())


class ECGFeatureClassificationDataset(Dataset):
    """(cached feature, multi-hot label) pairs for Step 3."""

    def __init__(self, df):
        self.df = df.reset_index() if "ecg_id" not in df.columns else df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ecg_id = row["ecg_id"] if "ecg_id" in row else row.name
        feats = np.load(_cache_path(ecg_id))          # (L, d)
        pooled = feats.mean(axis=0)                    # (d,)  simple mean-pool over time
        label = np.array(row["label_vector"], dtype=np.float32)
        return torch.tensor(pooled, dtype=torch.float32), torch.tensor(label)


class ECGFeatureReportDataset(Dataset):
    """(cached feature sequence, tokenized report) pairs for Step 4."""

    def __init__(self, df, tokenizer, max_len=config.MAX_REPORT_TOKENS, report_col="report"):
        self.df = df.reset_index() if "ecg_id" not in df.columns else df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.report_col = report_col
        # Drop rows with empty/NaN reports — Step 4 needs text targets
        self.df = self.df[self.df[report_col].notna() & (self.df[report_col].str.strip() != "")]
        self.df = self.df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ecg_id = row["ecg_id"] if "ecg_id" in row else row.name
        feats = np.load(_cache_path(ecg_id))  # (L, d)

        target_text = str(row[self.report_col])
        tok = self.tokenizer(
            target_text, max_length=self.max_len, truncation=True,
            padding="max_length", return_tensors="pt",
        )
        return {
            "encoder_features": torch.tensor(feats, dtype=torch.float32),
            "labels": tok["input_ids"].squeeze(0),
            "report_text": target_text,
        }


def collate_report_batch(batch):
    """Pad variable-length encoder feature sequences within a batch."""
    max_len = max(item["encoder_features"].shape[0] for item in batch)
    d = batch[0]["encoder_features"].shape[1]

    feats = torch.zeros(len(batch), max_len, d)
    attn_mask = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels = torch.stack([item["labels"] for item in batch])
    texts = [item["report_text"] for item in batch]

    for i, item in enumerate(batch):
        L = item["encoder_features"].shape[0]
        feats[i, :L] = item["encoder_features"]
        attn_mask[i, :L] = 1

    return {
        "encoder_features": feats,
        "attention_mask": attn_mask,
        "labels": labels,
        "report_text": texts,
    }

## 🏷️ Step 3 — Train the Classifier Head (Signal → Label)

Only the head's parameters are trained. The encoder stays frozen.  
Features are pre-computed once and cached to disk to speed up all epochs.


In [ ]:
"""
Step 3: Train a small classification head on top of frozen encoder features.

Usage:
    python -m src.classifier --train
    python -m src.classifier --evaluate
"""
import os
import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# from src import config  # (removed by generate_notebook.py — already in notebook scope)
# from src.data import load_full_dataset  # (removed by generate_notebook.py — already in notebook scope)
# from src.encoder import ECGEncoder  # (removed by generate_notebook.py — already in notebook scope)
# from src.datasets import ECGFeatureClassificationDataset, precompute_features  # (removed by generate_notebook.py — already in notebook scope)


class ClassifierHead(nn.Module):
    """One or two linear layers on top of the (pooled) encoder feature vector."""

    def __init__(self, in_dim=config.ENCODER_FEATURE_DIM,
                 hidden_dim=config.CLASSIFIER_HIDDEN_DIM,
                 num_classes=config.NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)  # raw logits; apply sigmoid outside for multi-label


def train():
    torch.manual_seed(config.RANDOM_SEED)
    data = load_full_dataset()
    encoder = ECGEncoder()

    print("Pre-computing (and caching) encoder features for all splits...")
    precompute_features(data["train"], encoder, desc="train features")
    precompute_features(data["val"], encoder, desc="val features")
    precompute_features(data["test"], encoder, desc="test features")

    train_ds = ECGFeatureClassificationDataset(data["train"])
    val_ds = ECGFeatureClassificationDataset(data["val"])
    train_loader = DataLoader(train_ds, batch_size=config.CLASSIFIER_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config.CLASSIFIER_BATCH_SIZE, shuffle=False)

    model = ClassifierHead(in_dim=encoder.feature_dim).to(config.DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.CLASSIFIER_LR)
    criterion = nn.BCEWithLogitsLoss()

    best_auroc = -1.0
    os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
    ckpt_path = os.path.join(config.CHECKPOINT_DIR, "classifier_head.pt")

    for epoch in range(config.CLASSIFIER_EPOCHS):
        model.train()
        total_loss = 0.0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.CLASSIFIER_EPOCHS}"):
            x, y = x.to(config.DEVICE), y.to(config.DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(train_ds)
        val_auroc = evaluate_loader(model, val_loader)
        print(f"Epoch {epoch+1}: train_loss={avg_loss:.4f}, val_macro_AUROC={val_auroc:.4f}")

        if val_auroc > best_auroc:
            best_auroc = val_auroc
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> New best model saved (AUROC={best_auroc:.4f}) to {ckpt_path}")


    print(f"Training done. Best val macro AUROC: {best_auroc:.4f}")


@torch.no_grad()
def evaluate_loader(model, loader, per_class=False):
    model.eval()
    all_logits, all_labels = [], []
    for x, y in loader:
        x = x.to(config.DEVICE)
        logits = model(x).cpu()
        all_logits.append(logits)
        all_labels.append(y)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-logits))  # sigmoid

    aurocs = []
    per_class_scores = {}
    for i, cls in enumerate(config.SUPERCLASSES):
        if len(np.unique(labels[:, i])) < 2:
            continue  # AUROC undefined if only one class present in this split
        score = roc_auc_score(labels[:, i], probs[:, i])
        aurocs.append(score)
        per_class_scores[cls] = score

    macro_auroc = float(np.mean(aurocs)) if aurocs else 0.0
    if per_class:
        return macro_auroc, per_class_scores
    return macro_auroc


def evaluate_test():
    data = load_full_dataset()
    encoder = ECGEncoder()
    precompute_features(data["test"], encoder, desc="test features")

    test_ds = ECGFeatureClassificationDataset(data["test"])
    test_loader = DataLoader(test_ds, batch_size=config.CLASSIFIER_BATCH_SIZE, shuffle=False)

    model = ClassifierHead(in_dim=encoder.feature_dim).to(config.DEVICE)
    ckpt_path = os.path.join(config.CHECKPOINT_DIR, "classifier_head.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"Classifier checkpoint not found at '{ckpt_path}'. "
            "Please train the model first using 'python -m src.classifier --train'."
        )
    model.load_state_dict(torch.load(ckpt_path, map_location=config.DEVICE))

    macro_auroc, per_class = evaluate_loader(model, test_loader, per_class=True)
    print(f"Test macro AUROC: {macro_auroc:.4f}")
    for cls, score in per_class.items():
        print(f"  {cls}: {score:.4f}")
    return macro_auroc, per_class


@torch.no_grad()
def predict_single(signal, encoder=None, model=None):
    """Used by the Gradio app: raw signal -> (label, confidence dict)."""
    if encoder is None:
        encoder = ECGEncoder()
    if model is None:
        model = ClassifierHead(in_dim=encoder.feature_dim).to(config.DEVICE)
        ckpt_path = os.path.join(config.CHECKPOINT_DIR, "classifier_head.pt")
        if os.path.exists(ckpt_path):
            model.load_state_dict(torch.load(ckpt_path, map_location=config.DEVICE))
        model.eval()

    feats = encoder.encode(signal)          # (L, d)
    pooled = feats.mean(dim=0, keepdim=True).to(config.DEVICE)  # (1, d)
    logits = model(pooled)
    probs = torch.sigmoid(logits).cpu().numpy()[0]
    return {cls: float(p) for cls, p in zip(config.SUPERCLASSES, probs)}



### 🚀 Run Step 3 Training

In [ ]:
# Pre-computes features (cached to FEATURES_CACHE_DIR) then trains for
# CLASSIFIER_EPOCHS epochs, saving the best checkpoint by val AUROC.
train()   # defined in the ClassifierHead cell above


### ✅ Step 3 Checkpoint — Test-set AUROC

In [ ]:
macro_auroc, per_class = evaluate_test()
print(f"\nTest macro AUROC: {macro_auroc:.4f}")
for cls, score in per_class.items():
    print(f"  {cls}: {score:.4f}")
print("\n✔  Step 3 CHECKPOINT PASSED")


## 📝 Step 4 — Train the Report Generator (Signal → Text)

Architecture: `HuBERT-ECG features → FeatureAdapter MLP → BART cross-attention → text`  
BART is fine-tuned with LoRA (lightweight, only ~2% of parameters are trainable).


In [ ]:
"""
Step 4: Adapter + BART decoder that turns encoder feature sequences into
free-text clinical reports.

The adapter is a small MLP that projects encoder feature vectors (dim d)
into BART's hidden size, then feeds them to BART as "encoder_outputs" so
BART's own cross-attention and decoder do the language generation.

Usage:
    python -m src.report_generator --train
    python -m src.report_generator --evaluate
    python -m src.report_generator --generate --ecg_id 1
"""
import os
import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BartForConditionalGeneration, BartTokenizerFast
from transformers.modeling_outputs import BaseModelOutput
from tqdm import tqdm

# from src import config  # (removed by generate_notebook.py — already in notebook scope)
# from src.data import load_full_dataset, load_raw_signal  # (removed by generate_notebook.py — already in notebook scope)
# from src.encoder import ECGEncoder  # (removed by generate_notebook.py — already in notebook scope)
# from src.datasets import ECGFeatureReportDataset, precompute_features, collate_report_batch  # (removed by generate_notebook.py — already in notebook scope)


class FeatureAdapter(nn.Module):
    """Projects (L, encoder_dim) ECG features into (L, bart_hidden_dim)."""

    def __init__(self, in_dim, out_dim, hidden_dim=config.ADAPTER_HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)  # (B, L, out_dim)


class ECGReportModel(nn.Module):
    """Wraps the adapter + BART so encoder features go in, token logits come out."""

    def __init__(self, encoder_dim, bart_model_id=config.BART_MODEL_ID, use_lora=config.USE_LORA):
        super().__init__()
        self.bart = BartForConditionalGeneration.from_pretrained(bart_model_id)
        bart_hidden = self.bart.config.d_model
        self.adapter = FeatureAdapter(encoder_dim, bart_hidden)

        if use_lora:
            try:
                from peft import LoraConfig, get_peft_model
                lora_cfg = LoraConfig(
                    r=16, lora_alpha=32, lora_dropout=0.1,
                    target_modules=["q_proj", "v_proj"],
                )
                self.bart = get_peft_model(self.bart, lora_cfg)
                print("LoRA applied to BART (lightweight fine-tuning).")
            except Exception as e:
                print(f"[WARNING] Could not apply LoRA ({e}); fine-tuning BART fully instead.")

    def forward(self, encoder_features, attention_mask, labels=None):
        adapted = self.adapter(encoder_features)  # (B, L, bart_hidden)
        encoder_outputs = BaseModelOutput(last_hidden_state=adapted)
        out = self.bart(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            labels=labels,
        )
        return out

    @torch.no_grad()
    def generate(self, encoder_features, attention_mask, tokenizer, num_beams=4, max_length=config.MAX_REPORT_TOKENS):
        adapted = self.adapter(encoder_features)
        encoder_outputs = BaseModelOutput(last_hidden_state=adapted)
        gen_ids = self.bart.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            num_beams=num_beams,
            max_length=max_length,
        )
        return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)


def train():
    torch.manual_seed(config.RANDOM_SEED)
    data = load_full_dataset()
    encoder = ECGEncoder()

    report_col = "report" if "report" in data["train"].columns else None
    if report_col is None:
        raise ValueError(
            "No 'report' column found in PTB-XL metadata. "
            "Check ptbxl_database.csv for the free-text report field name."
        )

    print("Pre-computing encoder features (reused from classifier cache if already run)...")
    precompute_features(data["train"], encoder, desc="train features")
    precompute_features(data["val"], encoder, desc="val features")

    tokenizer = BartTokenizerFast.from_pretrained(config.BART_MODEL_ID)

    train_ds = ECGFeatureReportDataset(data["train"], tokenizer, report_col=report_col)
    val_ds = ECGFeatureReportDataset(data["val"], tokenizer, report_col=report_col)
    train_loader = DataLoader(train_ds, batch_size=config.REPORT_GEN_BATCH_SIZE, shuffle=True,
                               collate_fn=collate_report_batch)
    val_loader = DataLoader(val_ds, batch_size=config.REPORT_GEN_BATCH_SIZE, shuffle=False,
                             collate_fn=collate_report_batch)

    model = ECGReportModel(encoder_dim=encoder.feature_dim).to(config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.REPORT_GEN_LR)

    best_val_loss = float("inf")
    os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
    ckpt_path = os.path.join(config.CHECKPOINT_DIR, "report_generator.pt")

    for epoch in range(config.REPORT_GEN_EPOCHS):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.REPORT_GEN_EPOCHS}"):
            feats = batch["encoder_features"].to(config.DEVICE)
            attn = batch["attention_mask"].to(config.DEVICE)
            labels = batch["labels"].to(config.DEVICE)
            labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)

            optimizer.zero_grad()
            out = model(feats, attn, labels=labels)
            out.loss.backward()
            optimizer.step()
            total_loss += out.loss.item() * feats.size(0)

        avg_train_loss = total_loss / len(train_ds)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                feats = batch["encoder_features"].to(config.DEVICE)
                attn = batch["attention_mask"].to(config.DEVICE)
                labels = batch["labels"].to(config.DEVICE)
                labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)
                out = model(feats, attn, labels=labels)
                val_loss += out.loss.item() * feats.size(0)
        avg_val_loss = val_loss / len(val_ds)

        print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> New best model saved (val_loss={best_val_loss:.4f}) to {ckpt_path}")


    print(f"Training done. Best val loss: {best_val_loss:.4f}")


def evaluate_test():
    """Compute BLEU / ROUGE on the held-out test split."""
    import evaluate as hf_evaluate

    data = load_full_dataset()
    encoder = ECGEncoder()
    report_col = "report"
    precompute_features(data["test"], encoder, desc="test features")

    tokenizer = BartTokenizerFast.from_pretrained(config.BART_MODEL_ID)
    test_ds = ECGFeatureReportDataset(data["test"], tokenizer, report_col=report_col)
    test_loader = DataLoader(test_ds, batch_size=config.REPORT_GEN_BATCH_SIZE, shuffle=False,
                              collate_fn=collate_report_batch)

    model = ECGReportModel(encoder_dim=encoder.feature_dim).to(config.DEVICE)
    ckpt_path = os.path.join(config.CHECKPOINT_DIR, "report_generator.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"Report generator checkpoint not found at '{ckpt_path}'. "
            "Please train the model first using 'python -m src.report_generator --train'."
        )
    model.load_state_dict(torch.load(ckpt_path, map_location=config.DEVICE))
    model.eval()

    bleu = hf_evaluate.load("bleu")
    rouge = hf_evaluate.load("rouge")

    predictions, references = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating reports"):
            feats = batch["encoder_features"].to(config.DEVICE)
            attn = batch["attention_mask"].to(config.DEVICE)
            preds = model.generate(feats, attn, tokenizer)
            predictions.extend(preds)
            references.extend(batch["report_text"])

    bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
    rouge_score = rouge.compute(predictions=predictions, references=references)

    print(f"BLEU: {bleu_score['bleu']:.4f}")
    print(f"ROUGE: {rouge_score}")

    # Save a side-by-side comparison for the "predicted vs real report" app feature
    out_path = os.path.join(config.OUTPUT_DIR, "test_predictions.csv")
    import pandas as pd
    pd.DataFrame({"predicted": predictions, "ground_truth": references}).to_csv(out_path, index=False)
    print(f"Saved predictions to {out_path}")

    return bleu_score, rouge_score


@torch.no_grad()
def generate_single(signal, encoder=None, model=None, tokenizer=None):
    """Used by the Gradio app: raw signal -> generated report string."""
    if encoder is None:
        encoder = ECGEncoder()
    if tokenizer is None:
        tokenizer = BartTokenizerFast.from_pretrained(config.BART_MODEL_ID)
    if model is None:
        model = ECGReportModel(encoder_dim=encoder.feature_dim).to(config.DEVICE)
        ckpt_path = os.path.join(config.CHECKPOINT_DIR, "report_generator.pt")
        if os.path.exists(ckpt_path):
            model.load_state_dict(torch.load(ckpt_path, map_location=config.DEVICE))
        model.eval()

    feats = encoder.encode(signal).unsqueeze(0).to(config.DEVICE)      # (1, L, d)
    attn = torch.ones(feats.shape[:2], dtype=torch.long).to(config.DEVICE)
    report = model.generate(feats, attn, tokenizer)[0]
    return report



### 🚀 Run Step 4 Training

In [ ]:
# Fine-tunes FeatureAdapter + LoRA-wrapped BART on (ECG features, report) pairs.
# Reuses cached features from Step 3 if already computed.
train()   # the train() defined in the Report Generator cell above
          # Note: if ClassifierHead.train() is also in scope, be explicit:
          # from the report_generator section, this is the correct one.


### ✅ Step 4 Checkpoint — BLEU & ROUGE + Sample Report

In [ ]:
bleu_score, rouge_score = evaluate_test()
print(f"\nBLEU : {bleu_score['bleu']:.4f}")
print(f"ROUGE: {rouge_score}")

# Generate a sample report for one test record
row    = data["test"].iloc[0]
signal, _ = load_raw_signal(row)
report = generate_single(signal)
print(f"\nGenerated report : {report}")
print(f"Ground truth      : {row.get('report', 'N/A')}")
print("\n✔  Step 4 CHECKPOINT PASSED")


## 📈 Step 6 — Systematic Evaluation & Error Analysis

Produces three output files:
- `per_diagnosis_performance.csv` — AUROC per class, sorted best → worst
- `failure_cases.csv`            — confidently-wrong test cases
- `evaluation_summary.txt`       — written paragraph summarising findings


In [ ]:
"""
Step 6: Systematic evaluation and error analysis.

Produces:
  outputs/per_diagnosis_performance.csv  - AUROC per class, sorted best-to-worst
  outputs/failure_cases.csv              - confidently-wrong test cases
  outputs/evaluation_summary.txt         - written paragraph-style summary

Usage:
    python -m src.evaluate_full
"""
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

# from src import config  # (removed by generate_notebook.py — already in notebook scope)
# from src.data import load_full_dataset  # (removed by generate_notebook.py — already in notebook scope)
# from src.encoder import ECGEncoder  # (removed by generate_notebook.py — already in notebook scope)
# from src.datasets import ECGFeatureClassificationDataset, precompute_features  # (removed by generate_notebook.py — already in notebook scope)
# from src.classifier import ClassifierHead  # (removed by generate_notebook.py — already in notebook scope)

# Diagnoses grouped by type, used for the "rhythm vs morphology" breakdown
# mentioned in the project plan.
# Note: With only the 5 PTB-XL superclasses (NORM/MI/STTC/CD/HYP), CD (conduction disturbance)
# is used as a stand-in proxy for rhythm/conduction. Expand to the full 71 SCP codes for explicit rhythm classes (e.g. AFIB).
DIAGNOSIS_TYPE = {
    "NORM": "baseline",
    "MI": "morphology",
    "STTC": "morphology",
    "CD": "rhythm/conduction",  # Stand-in proxy for rhythm/conduction under 5 superclasses
    "HYP": "morphology",
}


def run_classifier_error_analysis():
    data = load_full_dataset()
    encoder = ECGEncoder()
    precompute_features(data["test"], encoder, desc="test features")

    test_ds = ECGFeatureClassificationDataset(data["test"])
    test_loader = DataLoader(test_ds, batch_size=config.CLASSIFIER_BATCH_SIZE, shuffle=False)

    model = ClassifierHead(in_dim=encoder.feature_dim).to(config.DEVICE)
    ckpt_path = os.path.join(config.CHECKPOINT_DIR, "classifier_head.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"Classifier checkpoint not found at '{ckpt_path}'. "
            "Please train the model first using 'python -m src.classifier --train'."
        )
    model.load_state_dict(torch.load(ckpt_path, map_location=config.DEVICE))
    model.eval()

    all_logits, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            logits = model(x.to(config.DEVICE)).cpu()
            all_logits.append(logits)
            all_labels.append(y)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    # --- Per-diagnosis performance table ---
    rows = []
    for i, cls in enumerate(config.SUPERCLASSES):
        if len(np.unique(labels[:, i])) < 2:
            continue
        auroc = roc_auc_score(labels[:, i], probs[:, i])
        rows.append({
            "diagnosis": cls,
            "type": DIAGNOSIS_TYPE.get(cls, "unknown"),
            "AUROC": round(auroc, 4),
            "n_positive": int(labels[:, i].sum()),
        })
    perf_df = pd.DataFrame(rows).sort_values("AUROC", ascending=False)
    perf_path = os.path.join(config.OUTPUT_DIR, "per_diagnosis_performance.csv")
    perf_df.to_csv(perf_path, index=False)
    print(f"Saved per-diagnosis performance table to {perf_path}")
    print(perf_df.to_string(index=False))

    # --- Failure case list: confidently wrong predictions ---
    test_ids = test_ds.df["ecg_id"].values if "ecg_id" in test_ds.df.columns else test_ds.df.index.values
    failure_rows = []
    for row_idx in range(len(labels)):
        wrong_mask = preds[row_idx] != labels[row_idx]
        if not wrong_mask.any():
            continue
        confidence = np.max(np.abs(probs[row_idx] - 0.5)) * 2  # 0..1, how far from the decision boundary
        if confidence < 0.5:
            continue  # only keep "confidently wrong" cases
        true_classes = [config.SUPERCLASSES[i] for i in range(config.NUM_CLASSES) if labels[row_idx, i] == 1]
        pred_classes = [config.SUPERCLASSES[i] for i in range(config.NUM_CLASSES) if preds[row_idx, i] == 1]
        failure_rows.append({
            "ecg_id": test_ids[row_idx],
            "true_diagnosis": ", ".join(true_classes) or "none",
            "predicted_diagnosis": ", ".join(pred_classes) or "none",
            "confidence": round(float(confidence), 3),
        })
    failure_df = pd.DataFrame(failure_rows).sort_values("confidence", ascending=False)
    failure_path = os.path.join(config.OUTPUT_DIR, "failure_cases.csv")
    failure_df.to_csv(failure_path, index=False)
    print(f"\nSaved {len(failure_df)} confidently-wrong failure cases to {failure_path}")

    # --- Written summary paragraph ---
    if len(perf_df) > 0:
        best = perf_df.iloc[0]
        worst = perf_df.iloc[-1]
        rhythm_scores = perf_df[perf_df.type == "rhythm/conduction"]["AUROC"]
        morph_scores = perf_df[perf_df.type == "morphology"]["AUROC"]
        summary = (
            f"Evaluation summary\n"
            f"===================\n\n"
            f"The classifier performs best on '{best.diagnosis}' (AUROC={best.AUROC}) "
            f"and worst on '{worst.diagnosis}' (AUROC={worst.AUROC}).\n\n"
        )
        if len(rhythm_scores) and len(morph_scores):
            summary += (
                f"Rhythm/conduction diagnoses (CD proxy) averaged AUROC={rhythm_scores.mean():.4f}, "
                f"while morphology-defined diagnoses averaged AUROC={morph_scores.mean():.4f}. "
            )
            if rhythm_scores.mean() > morph_scores.mean():
                summary += (
                    "This matches the pattern described in the project plan: rhythm abnormalities "
                    "tend to be easier to detect from waveform shape alone, while morphology-based "
                    "diagnoses (e.g. infarction, hypertrophy) require finer-grained amplitude/interval "
                    "cues that a frozen, general-purpose encoder may under-represent. "
                    "(Note: CD serves as a proxy under the 5 superclasses; full 71 SCP codes allow pure rhythm evaluation).\n\n"
                )
            else:
                summary += "\n\n"
        summary += (
            f"{len(failure_df)} confidently-wrong test cases were identified "
            f"(confidence > 0.5 away from the decision boundary, prediction wrong). "
            f"See failure_cases.csv for the full list and per_diagnosis_performance.csv "
            f"for the sorted performance table.\n"
        )
        summary_path = os.path.join(config.OUTPUT_DIR, "evaluation_summary.txt")
        with open(summary_path, "w") as f:
            f.write(summary)
        print(f"\nSaved written summary to {summary_path}")
        print("\n" + summary)



### ✅ Step 6 Checkpoint — Run Full Evaluation

In [ ]:
run_classifier_error_analysis()

# Display the performance table inline
import pandas as pd
perf_df = pd.read_csv(os.path.join(OUTPUT_DIR, "per_diagnosis_performance.csv"))
print("\nPer-Diagnosis Performance (sorted best → worst):")
print(perf_df.to_string(index=False))

fail_df = pd.read_csv(os.path.join(OUTPUT_DIR, "failure_cases.csv"))
print(f"\nTotal confidently-wrong failure cases: {len(fail_df)}")
print(fail_df.head(10).to_string(index=False))

with open(os.path.join(OUTPUT_DIR, "evaluation_summary.txt")) as f:
    print("\n" + f.read())

print("✔  Step 6 CHECKPOINT PASSED")


## 🌐 Step 5 — Launch Gradio Web Application

Runs the interactive ECG analysis app.  
A **public `gradio.live` URL** is printed below — open it in any browser.  
The link is valid for **72 hours**.

> ⚠️ Keep this cell running. The server dies when the cell stops.


In [ ]:
import os
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
import torch
from transformers import BartTokenizerFast

print("Loading models (this happens once at startup)...")
_data      = load_full_dataset()
_test_df   = _data["test"]
_encoder   = ECGEncoder()

_classifier = ClassifierHead(in_dim=_encoder.feature_dim).to(DEVICE)
_clf_ckpt   = os.path.join(CHECKPOINT_DIR, "classifier_head.pt")
if os.path.exists(_clf_ckpt):
    _classifier.load_state_dict(torch.load(_clf_ckpt, map_location=DEVICE))
_classifier.eval()

_tokenizer    = BartTokenizerFast.from_pretrained(BART_MODEL_ID)
_report_model = ECGReportModel(encoder_dim=_encoder.feature_dim).to(DEVICE)
_report_ckpt  = os.path.join(CHECKPOINT_DIR, "report_generator.pt")
if os.path.exists(_report_ckpt):
    _report_model.load_state_dict(torch.load(_report_ckpt, map_location=DEVICE))
_report_model.eval()

_record_choices = [str(i) for i in _test_df.index[:200]]


def _plot_to_image(signal, title):
    lead_names = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
    fig, axes = plt.subplots(12, 1, figsize=(9, 11), sharex=True)
    for i, ax in enumerate(axes):
        ax.plot(signal[:, i], linewidth=0.7, color="#1a1a1a")
        ax.set_ylabel(lead_names[i], rotation=0, labelpad=18, fontsize=8)
        ax.set_yticks([])
    axes[-1].set_xlabel("Samples")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def analyze_record(ecg_id_str):
    ecg_id = int(ecg_id_str)
    row    = _test_df.loc[ecg_id]
    signal, _ = load_raw_signal(row)
    fig    = _plot_to_image(signal, title=f"Record {ecg_id}")
    confidences = predict_single(signal, encoder=_encoder, model=_classifier)
    top_label   = max(confidences, key=confidences.get)
    top_conf    = confidences[top_label]
    label_str   = f"**{top_label}** ({top_conf*100:.1f}% confidence)"
    conf_table  = "\n".join(
        [f"- {k}: {v*100:.1f}%" for k, v in sorted(confidences.items(), key=lambda kv: -kv[1])]
    )
    generated_report = generate_single(signal, encoder=_encoder,
                                       model=_report_model, tokenizer=_tokenizer)
    ground_truth = str(row.get("report", "N/A"))
    side_by_side = (
        f"**Generated report:**\n{generated_report}\n\n"
        f"**Cardiologist ground truth:**\n{ground_truth}"
    )
    return fig, label_str, conf_table, side_by_side


def analyze_uploaded_csv(file_obj):
    if file_obj is None:
        return None, "No file uploaded.", "", ""
    signal = np.loadtxt(file_obj.name, delimiter=",")
    if signal.shape[1] != N_LEADS:
        return None, f"Expected {N_LEADS} columns, got {signal.shape[1]}.", "", ""
    fig = _plot_to_image(signal, title="Uploaded ECG")
    confidences = predict_single(signal, encoder=_encoder, model=_classifier)
    top_label   = max(confidences, key=confidences.get)
    top_conf    = confidences[top_label]
    label_str   = f"**{top_label}** ({top_conf*100:.1f}% confidence)"
    conf_table  = "\n".join(
        [f"- {k}: {v*100:.1f}%" for k, v in sorted(confidences.items(), key=lambda kv: -kv[1])]
    )
    generated_report = generate_single(signal, encoder=_encoder,
                                       model=_report_model, tokenizer=_tokenizer)
    return fig, label_str, conf_table, f"**Generated report:**\n{generated_report}"


with gr.Blocks(title="Intelligent ECG Analysis Tool") as demo:
    gr.Markdown(
        "# 🫀 Intelligent ECG Analysis Tool\n"
        "Signal-to-Report and Signal-to-Diagnosis with Deep Learning.\n\n"
        "Pick a record from the PTB-XL test set, or upload your own 12-lead CSV."
    )
    with gr.Tab("Test-set record"):
        with gr.Row():
            dropdown = gr.Dropdown(
                choices=_record_choices, label="Select a test-set ECG (record ID)",
                value=_record_choices[0] if _record_choices else None
            )
            run_btn = gr.Button("Analyze", variant="primary")
        with gr.Row():
            plot_out  = gr.Plot(label="12-lead signal")
            with gr.Column():
                label_out = gr.Markdown(label="Diagnosis")
                conf_out  = gr.Markdown(label="Confidence (all classes)")
        report_out = gr.Markdown(label="Report comparison")
        run_btn.click(analyze_record, inputs=dropdown,
                      outputs=[plot_out, label_out, conf_out, report_out])

    with gr.Tab("Upload your own"):
        gr.Markdown("CSV with shape (n_samples, 12), one column per lead, no header.")
        file_in    = gr.File(label="Upload CSV", file_types=[".csv"])
        upload_btn = gr.Button("Analyze uploaded ECG", variant="primary")
        with gr.Row():
            plot_out2  = gr.Plot(label="12-lead signal")
            with gr.Column():
                label_out2 = gr.Markdown(label="Diagnosis")
                conf_out2  = gr.Markdown(label="Confidence (all classes)")
        report_out2 = gr.Markdown(label="Generated report")
        upload_btn.click(analyze_uploaded_csv, inputs=file_in,
                         outputs=[plot_out2, label_out2, conf_out2, report_out2])

    gr.Markdown(
        "---\n"
        "*Research/educational prototype. Not a certified diagnostic device.*"
    )

# Auto-detect Kaggle/Colab → share=True for public URL
on_cloud = os.path.exists("/kaggle/input") or os.path.exists("/content")
demo.launch(share=on_cloud)
